In [1]:
import sys
from pathlib import Path
import logging
from pathlib import Path
from typing import Any, Dict, List, Optional

# 1. Point to the 'backend' folder (the parent of 'src')
backend_path = Path.cwd().resolve()
while backend_path.name != "backend" and backend_path.parent != backend_path:
    backend_path = backend_path.parent

if str(backend_path) not in sys.path:
    sys.path.insert(0, str(backend_path))

print("Added to sys.path:", backend_path)

# 2. Always import with the 'src.' prefix so package hierarchies match
from src.config import DATA_DIR, logger
from src.ingestion import (
    CanonicalRecord,
    execute_incremental_ingestion_batch,
    process_file_with_validation,
)
from src.schema import SCHEMA_CONFIG
from src.agent.state import WikiState
from src.agent.nodes.collect_data import collect_data
import operator
from typing import Annotated, Any, Dict, List, Optional, TypedDict


Added to sys.path: C:\Users\ps302\OneDrive\Desktop\garph hacks\backend


In [2]:
def create_initial_wiki_state(
    chunk_size: int = 500,
    files_to_process: Optional[List[str]] = None,
    previous_hashes: Optional[Dict[str, str]] = None,
    max_workers: int = 4,
    data_dir: Optional[str] = None,
    batch_id: Optional[str] = None,
    reset_manifest: bool = False,
    query_text: Optional[str] = None,
    target_entity_id: Optional[str] = None,
) -> WikiState:
    """Helper to initialize an empty WikiState with defaults."""
    return {
        "reset_manifest": reset_manifest,
        "batch_id": batch_id or "",
        "chunk_size": chunk_size,
        "data_dir": data_dir,
        "max_workers": max_workers,
        "files_to_process": files_to_process or [],
        "previous_hashes": previous_hashes or {},
        "current_hashes": {},
        "changed_entities": {"added": [], "updated": [], "unchanged": []},
        "quarantined_records": [],
        "unresolved_foreign_keys": {},
        "now_resolved_foreign_keys": {},
        "canonical_records": {},
        "relationships_records": [],
        "dirty_entity_ids": [],
        "audit_report": {},
        "branch_status": {},
        "chunk_metadata": [],
        "stats": {},
        "errors": [],
        "query_text": query_text,
        "target_entity_id": target_entity_id,
        "query_result": {},
        "stage": "initialized",
    }


In [3]:
state = create_initial_wiki_state(
    files_to_process=["departments.csv", ]    #"teams.csv", "employees.csv"
)


In [4]:

def merge_canonical_records(
    left: Optional[Dict[str, List[Dict[str, Any]]]], 
    right: Optional[Dict[str, List[Dict[str, Any]]]]
) -> Dict[str, List[Dict[str, Any]]]:
    """
    Reducer that merges canonical records grouped by entity_type,
    deduplicating/updating by entity_id.
    """
    if not left:
        return dict(right or {})
    if not right:
        return dict(left)

    result = {k: list(v) for k, v in left.items()}
    for entity_type, new_records in right.items():
        if entity_type not in result:
            result[entity_type] = list(new_records)
        else:
            existing_map = {rec["entity_id"]: idx for idx, rec in enumerate(result[entity_type])}
            for rec in new_records:
                eid = rec.get("entity_id")
                if eid in existing_map:
                    result[entity_type][existing_map[eid]] = rec
                else:
                    result[entity_type].append(rec)
                    existing_map[eid] = len(result[entity_type]) - 1
    return result


def merge_dicts(
    left: Optional[Dict[str, Any]], 
    right: Optional[Dict[str, Any]]
) -> Dict[str, Any]:
    """Reducer that merges dictionary keys safely during parallel execution."""
    merged = dict(left or {})
    if right:
        merged.update(right)
    return merged


def take_latest(left: Any, right: Any) -> Any:
    """Reducer that accepts the latest updated value."""
    return right if right is not None else left


class WikiState(TypedDict, total=False):
    """
    Complete state passed between nodes in the Company Brain LangGraph pipeline.
    All list and dict fields used in parallel branches have reducer annotations.
    """
    # Ingestion Configuration
    reset_manifest: Optional[bool]
    batch_id: Annotated[str, take_latest]
    chunk_size: int
    data_dir: Optional[str]
    max_workers: int
    files_to_process: List[str]

    # Change Detection & Incremental Watermarks
    previous_hashes: Dict[str, str]
    current_hashes: Annotated[Dict[str, str], merge_dicts]
    changed_entities: Dict[str, List[str]]  # {"added": [...], "updated": [...], "unchanged": [...]}
    quarantined_records: Annotated[List[Dict[str, Any]], operator.add]
    
    # Foreign Key Resolution Queues
    unresolved_foreign_keys: Annotated[Dict[str, Any], merge_dicts]
    now_resolved_foreign_keys: Annotated[Dict[str, Any], merge_dicts]

    # Ingested Canonical Records and Edges
    canonical_records: Annotated[Dict[str, List[Dict[str, Any]]], merge_canonical_records]
    relationships_records: Annotated[List[Dict[str, Any]], operator.add]

    # Dirty Entity IDs for Incremental Wiki Generation
    dirty_entity_ids: Annotated[List[str], take_latest]

    # Audit & Status
    audit_report: Annotated[Dict[str, Any], merge_dicts]
    branch_status: Annotated[Dict[str, str], merge_dicts]

    # Observability & Chunk Metadata
    chunk_metadata: Annotated[List[Dict[str, Any]], operator.add]
    stats: Annotated[Dict[str, Any], merge_dicts]
    errors: Annotated[List[str], operator.add]

    # Retrieval Interface
    query_text: Optional[str]
    target_entity_id: Optional[str]
    query_result: Annotated[Dict[str, Any], merge_dicts]

    # Pipeline Stage Tracking
    stage: Annotated[str, take_latest]


In [5]:
def collect_data(state: WikiState) -> Dict[str, Any]:
    """
    LangGraph Node: Collect and parse company data in chunks across files with Step 1 validation.

    Performs:
    1. Checkpoint & batch ID initialization.
    2. In-chunk conflict & duplicate quarantine.
    3. Delta classification: added, updated (with field diffs), unchanged.
    4. Detection of missing parents / unresolved foreign-keys.
    5. Storage into WikiState for downstream graph synchronization.
    """
    logger.info("--- [LangGraph Node: collect_data] Executing Step 1 Ingestion Pipeline ---")

    chunk_size = state.get("chunk_size", 500)
    max_workers = state.get("max_workers", 4)
    data_dir_str = state.get("data_dir")
    data_dir = Path(data_dir_str) if data_dir_str else DATA_DIR
    batch_id = state.get("batch_id")

    # Dynamically determine files to process
    files_to_process = state.get("files_to_process")
    if not files_to_process:
        files_to_process = list(SCHEMA_CONFIG.keys())
        if (data_dir / "relationships.csv").exists():
            files_to_process.append("relationships.csv")

    logger.info(f"Ingesting {len(files_to_process)} CSV files with chunk_size={chunk_size}, workers={max_workers}")

    previous_hashes = state.get("previous_hashes")
    reset_manifest = state.get("reset_manifest", False)

    # Run the Step 1 validated batch ingestion engine
    batch_result = execute_incremental_ingestion_batch(
        filenames=files_to_process,
        data_dir=data_dir,
        chunk_size=chunk_size,
        batch_id=batch_id,
        previous_hashes=previous_hashes,
        max_workers=max_workers,
        persist_manifest=True,
        reset_manifest=reset_manifest,
    )

    records_by_type = batch_result["records_by_type"]
    relationships = batch_result["relationships"]
    all_hashes = batch_result["current_hashes"]
    quarantined = batch_result["quarantined"]
    unresolved_fks = batch_result["unresolved_foreign_keys"]
    now_resolved_fks = batch_result["now_resolved_foreign_keys"]
    chunk_meta = batch_result["chunk_metadata"]
    errors = batch_result["errors"]
    stats = batch_result["stats"]

    # Compute categorized list of changed entity IDs
    added_ids: List[str] = []
    updated_ids: List[str] = []
    unchanged_ids: List[str] = []

    for label, records in records_by_type.items():
        for rec in records:
            eid = rec["entity_id"]
            ctype = rec.get("change_type", "added")
            if ctype == "added":
                added_ids.append(eid)
            elif ctype == "updated":
                updated_ids.append(eid)
            else:
                unchanged_ids.append(eid)

    changed_summary = {
        "added": added_ids,
        "updated": updated_ids,
        "unchanged": unchanged_ids,
    }

    logger.info(
        f"Data collection complete [Batch {batch_result['batch_id']}]: "
        f"{stats['total_canonical_entities']} entities, {stats['total_relationships']} edges. "
        f"(Added: {len(added_ids)}, Updated: {len(updated_ids)}, Unchanged: {len(unchanged_ids)}, "
        f"Quarantined: {len(quarantined)}, Unresolved FKs: {len(unresolved_fks)})"
    )

    # Return partial state update for LangGraph reducers
    return {
        "batch_id": batch_result["batch_id"],
        "canonical_records": records_by_type,
        "relationships_records": relationships,
        "current_hashes": all_hashes,
        "changed_entities": changed_summary,
        "quarantined_records": quarantined,
        "unresolved_foreign_keys": unresolved_fks,
        "now_resolved_foreign_keys": now_resolved_fks,
        "chunk_metadata": chunk_meta,
        "stats": stats,
        "errors": errors,
        "stage": "data_collected",
    }


In [6]:
out = collect_data(state)

2026-10-16 16:46:04,194 [INFO] company_brain: --- [LangGraph Node: collect_data] Executing Step 1 Ingestion Pipeline ---
2026-10-16 16:46:04,195 [INFO] company_brain: Ingesting 1 CSV files with chunk_size=500, workers=4
2026-10-16 16:46:04,213 [INFO] company_brain: Starting Ingestion Batch [batch_20261016_111604_ccf831f3]: 1 files (chunk_size=500)
2026-10-16 16:46:04,233 [INFO] company_brain: Batch [batch_20261016_111604_ccf831f3] complete: 12 entities, 0 edges. (Added: 12, Updated: 0, Unchanged: 0, Quarantined: 0)
2026-10-16 16:46:04,234 [INFO] company_brain: Data collection complete [Batch batch_20261016_111604_ccf831f3]: 12 entities, 0 edges. (Added: 12, Updated: 0, Unchanged: 0, Quarantined: 0, Unresolved FKs: 0)


In [11]:
out

{'batch_id': 'batch_20261016_111604_ccf831f3',
 'canonical_records': {'Department': [{'entity_type': 'Department',
    'entity_id': 'DEPT-ENG',
    'display_name': 'Engineering',
    'properties': {'department_id': 'DEPT-ENG',
     'department_name': 'Engineering',
     'mission': 'Build and operate reliable product software',
     'head_employee_id': 'EMP-0001',
     'created_at': '2025-01-01T09:00:00Z'},
    'foreign_keys': {'head_employee_id': ['Employee',
      'employee_id',
      'EMP-0001']},
    'source_file': 'departments.csv',
    'row_hash': '64773914a0c7c31f',
    'chunk_index': 0,
    'change_type': 'added',
    'previous_hash': None,
    'changed_fields': [],
    'previous_values': {},
    'unresolved_fks': {},
    'is_tombstone': False},
   {'entity_type': 'Department',
    'entity_id': 'DEPT-PROD',
    'display_name': 'Product',
    'properties': {'department_id': 'DEPT-PROD',
     'department_name': 'Product',
     'mission': 'Define customer problems, roadmap, and pro

In [12]:
import ast
import json

# Put your raw string payload here
raw_string = """
{'batch_id': 'batch_20261016_111604_ccf831f3',
 'canonical_records': {'Department': [{'entity_type': 'Department',
    'entity_id': 'DEPT-ENG',
    'display_name': 'Engineering',
    'properties': {'department_id': 'DEPT-ENG',
     'department_name': 'Engineering',
     'mission': 'Build and operate reliable product software',
     'head_employee_id': 'EMP-0001',
     'created_at': '2025-01-01T09:00:00Z'},
    'foreign_keys': {'head_employee_id': ['Employee',
      'employee_id',
      'EMP-0001']},
    'source_file': 'departments.csv',
    'row_hash': '64773914a0c7c31f',
    'chunk_index': 0,
    'change_type': 'added',
    'previous_hash': None,
    'changed_fields': [],
    'previous_values': {},
    'unresolved_fks': {},
    'is_tombstone': False},
   {'entity_type': 'Department',
    'entity_id': 'DEPT-PROD',
    'display_name': 'Product',
    'properties': {'department_id': 'DEPT-PROD',
     'department_name': 'Product',
     'mission': 'Define customer problems, roadmap, and product outcomes',
     'head_employee_id': 'EMP-0002',
     'created_at': '2025-01-01T09:00:00Z'},
    'foreign_keys': {'head_employee_id': ['Employee',
      'employee_id',
      'EMP-0002']},
    'source_file': 'departments.csv',
    'row_hash': 'd11cbc4fe62921b0',
    'chunk_index': 0,
    'change_type': 'added',
    'previous_hash': None,
    'changed_fields': [],
    'previous_values': {},
    'unresolved_fks': {},
    'is_tombstone': False},
   {'entity_type': 'Department',
    'entity_id': 'DEPT-DATA',
    'display_name': 'Data & AI',
    'properties': {'department_id': 'DEPT-DATA',
     'department_name': 'Data & AI',
     'mission': 'Build analytics, ML, and AI-powered capabilities',
     'head_employee_id': 'EMP-0003',
     'created_at': '2025-01-01T09:00:00Z'},
    'foreign_keys': {'head_employee_id': ['Employee',
      'employee_id',
      'EMP-0003']},
    'source_file': 'departments.csv',
    'row_hash': '9445dad2d5af98c5',
    'chunk_index': 0,
    'change_type': 'added',
    'previous_hash': None,
    'changed_fields': [],
    'previous_values': {},
    'unresolved_fks': {},
    'is_tombstone': False},
   {'entity_type': 'Department',
    'entity_id': 'DEPT-DESIGN',
    'display_name': 'Design & Research',
    'properties': {'department_id': 'DEPT-DESIGN',
     'department_name': 'Design & Research',
     'mission': 'Create accessible, usable product experiences',
     'head_employee_id': 'EMP-0004',
     'created_at': '2025-01-01T09:00:00Z'},
    'foreign_keys': {'head_employee_id': ['Employee',
      'employee_id',
      'EMP-0004']},
    'source_file': 'departments.csv',
    'row_hash': 'ee43309b9fe87755',
    'chunk_index': 0,
    'change_type': 'added',
    'previous_hash': None,
    'changed_fields': [],
    'previous_values': {},
    'unresolved_fks': {},
    'is_tombstone': False},
   {'entity_type': 'Department',
    'entity_id': 'DEPT-SALES',
    'display_name': 'Sales',
    'properties': {'department_id': 'DEPT-SALES',
     'department_name': 'Sales',
     'mission': 'Acquire customers and manage the sales pipeline',
     'head_employee_id': 'EMP-0005',
     'created_at': '2025-01-01T09:00:00Z'},
    'foreign_keys': {'head_employee_id': ['Employee',
      'employee_id',
      'EMP-0005']},
    'source_file': 'departments.csv',
    'row_hash': '8104a99d1c8c3ad8',
    'chunk_index': 0,
    'change_type': 'added',
    'previous_hash': None,
    'changed_fields': [],
    'previous_values': {},
    'unresolved_fks': {},
    'is_tombstone': False},
   {'entity_type': 'Department',
    'entity_id': 'DEPT-MKT',
    'display_name': 'Marketing',
    'properties': {'department_id': 'DEPT-MKT',
     'department_name': 'Marketing',
     'mission': 'Position the product and generate qualified demand',
     'head_employee_id': 'EMP-0006',
     'created_at': '2025-01-01T09:00:00Z'},
    'foreign_keys': {'head_employee_id': ['Employee',
      'employee_id',
      'EMP-0006']},
    'source_file': 'departments.csv',
    'row_hash': 'ca9b7bc1c200419b',
    'chunk_index': 0,
    'change_type': 'added',
    'previous_hash': None,
    'changed_fields': [],
    'previous_values': {},
    'unresolved_fks': {},
    'is_tombstone': False},
   {'entity_type': 'Department',
    'entity_id': 'DEPT-CS',
    'display_name': 'Customer Success',
    'properties': {'department_id': 'DEPT-CS',
     'department_name': 'Customer Success',
     'mission': 'Drive adoption, retention, and customer outcomes',
     'head_employee_id': 'EMP-0007',
     'created_at': '2025-01-01T09:00:00Z'},
    'foreign_keys': {'head_employee_id': ['Employee',
      'employee_id',
      'EMP-0007']},
    'source_file': 'departments.csv',
    'row_hash': 'eed4d418b29a185e',
    'chunk_index': 0,
    'change_type': 'added',
    'previous_hash': None,
    'changed_fields': [],
    'previous_values': {},
    'unresolved_fks': {},
    'is_tombstone': False},
   {'entity_type': 'Department',
    'entity_id': 'DEPT-SUPPORT',
    'display_name': 'Customer Support',
    'properties': {'department_id': 'DEPT-SUPPORT',
     'department_name': 'Customer Support',
     'mission': 'Resolve customer questions and product issues',
     'head_employee_id': 'EMP-0008',
     'created_at': '2025-01-01T09:00:00Z'},
    'foreign_keys': {'head_employee_id': ['Employee',
      'employee_id',
      'EMP-0008']},
    'source_file': 'departments.csv',
    'row_hash': '0734d09842bd3104',
    'chunk_index': 0,
    'change_type': 'added',
    'previous_hash': None,
    'changed_fields': [],
    'previous_values': {},
    'unresolved_fks': {},
    'is_tombstone': False},
   {'entity_type': 'Department',
    'entity_id': 'DEPT-OPS',
    'display_name': 'Business Operations',
    'properties': {'department_id': 'DEPT-OPS',
     'department_name': 'Business Operations',
     'mission': 'Improve internal processes and operating cadence',
     'head_employee_id': 'EMP-0009',
     'created_at': '2025-01-01T09:00:00Z'},
    'foreign_keys': {'head_employee_id': ['Employee',
      'employee_id',
      'EMP-0009']},
    'source_file': 'departments.csv',
    'row_hash': 'e42f3ff95f3475a4',
    'chunk_index': 0,
    'change_type': 'added',
    'previous_hash': None,
    'changed_fields': [],
    'previous_values': {},
    'unresolved_fks': {},
    'is_tombstone': False},
   {'entity_type': 'Department',
    'entity_id': 'DEPT-PEOPLE',
    'display_name': 'People & Talent',
    'properties': {'department_id': 'DEPT-PEOPLE',
     'department_name': 'People & Talent',
     'mission': 'Recruit, develop, and support employees',
     'head_employee_id': 'EMP-0010',
     'created_at': '2025-01-01T09:00:00Z'},
    'foreign_keys': {'head_employee_id': ['Employee',
      'employee_id',
      'EMP-0010']},
    'source_file': 'departments.csv',
    'row_hash': '52e0bdd494b38efb',
    'chunk_index': 0,
    'change_type': 'added',
    'previous_hash': None,
    'changed_fields': [],
    'previous_values': {},
    'unresolved_fks': {},
    'is_tombstone': False},
   {'entity_type': 'Department',
    'entity_id': 'DEPT-FIN',
    'display_name': 'Finance & Legal',
    'properties': {'department_id': 'DEPT-FIN',
     'department_name': 'Finance & Legal',
     'mission': 'Manage finance, contracts, and compliance',
     'head_employee_id': 'EMP-0011',
     'created_at': '2025-01-01T09:00:00Z'},
    'foreign_keys': {'head_employee_id': ['Employee',
      'employee_id',
      'EMP-0011']},
    'source_file': 'departments.csv',
    'row_hash': 'b7f1651f12206a51',
    'chunk_index': 0,
    'change_type': 'added',
    'previous_hash': None,
    'changed_fields': [],
    'previous_values': {},
    'unresolved_fks': {},
    'is_tombstone': False},
   {'entity_type': 'Department',
    'entity_id': 'DEPT-SEC',
    'display_name': 'Security & IT',
    'properties': {'department_id': 'DEPT-SEC',
     'department_name': 'Security & IT',
     'mission': 'Protect company systems, access, and data',
     'head_employee_id': 'EMP-0012',
     'created_at': '2025-01-01T09:00:00Z'},
    'foreign_keys': {'head_employee_id': ['Employee',
      'employee_id',
      'EMP-0012']},
    'source_file': 'departments.csv',
    'row_hash': '01bb4bcf97ef8caa',
    'chunk_index': 0,
    'change_type': 'added',
    'previous_hash': None,
    'changed_fields': [],
    'previous_values': {},
    'unresolved_fks': {},
    'is_tombstone': False}]},
 'relationships_records': [],
 'current_hashes': {'DEPT-ENG': '64773914a0c7c31f',
  'DEPT-PROD': 'd11cbc4fe62921b0',
  'DEPT-DATA': '9445dad2d5af98c5',
  'DEPT-DESIGN': 'ee43309b9fe87755',
  'DEPT-SALES': '8104a99d1c8c3ad8',
  'DEPT-MKT': 'ca9b7bc1c200419b',
  'DEPT-CS': 'eed4d418b29a185e',
  'DEPT-SUPPORT': '0734d09842bd3104',
  'DEPT-OPS': 'e42f3ff95f3475a4',
  'DEPT-PEOPLE': '52e0bdd494b38efb',
  'DEPT-FIN': 'b7f1651f12206a51',
  'DEPT-SEC': '01bb4bcf97ef8caa'},
 'changed_entities': {'added': ['DEPT-ENG',
   'DEPT-PROD',
   'DEPT-DATA',
   'DEPT-DESIGN',
   'DEPT-SALES',
   'DEPT-MKT',
   'DEPT-CS',
   'DEPT-SUPPORT',
   'DEPT-OPS',
   'DEPT-PEOPLE',
   'DEPT-FIN',
   'DEPT-SEC'],
  'updated': [],
  'unchanged': []},
 'quarantined_records': [],
 'unresolved_foreign_keys': {},
 'now_resolved_foreign_keys': {},
 'chunk_metadata': [{'filename': 'departments.csv',
   'label': 'Department',
   'total_rows': 12,
   'chunks': 1,
   'added': 12,
   'updated': 0,
   'unchanged': 0,
   'quarantined': 0}],
 'stats': {'batch_id': 'batch_20261016_111604_ccf831f3',
  'total_files': 1,
  'total_canonical_entities': 12,
  'total_relationships': 0,
  'total_chunks': 1,
  'added': 12,
  'updated': 0,
  'unchanged': 0,
  'quarantined': 0},
 'errors': [],
 'stage': 'data_collected'}"""


# 1. Safely evaluate the string as a Python dictionary
# This handles the single quotes, False, True, and None automatically
python_dict = ast.literal_eval(raw_string)

# 2. Convert the dictionary into a perfectly valid JSON string
valid_json = json.dumps(python_dict, indent=2)

print(valid_json)

{
  "batch_id": "batch_20261016_111604_ccf831f3",
  "canonical_records": {
    "Department": [
      {
        "entity_type": "Department",
        "entity_id": "DEPT-ENG",
        "display_name": "Engineering",
        "properties": {
          "department_id": "DEPT-ENG",
          "department_name": "Engineering",
          "mission": "Build and operate reliable product software",
          "head_employee_id": "EMP-0001",
          "created_at": "2025-01-01T09:00:00Z"
        },
        "foreign_keys": {
          "head_employee_id": [
            "Employee",
            "employee_id",
            "EMP-0001"
          ]
        },
        "source_file": "departments.csv",
        "row_hash": "64773914a0c7c31f",
        "chunk_index": 0,
        "change_type": "added",
        "previous_hash": null,
        "changed_fields": [],
        "previous_values": {},
        "unresolved_fks": {},
        "is_tombstone": false
      },
      {
        "entity_type": "Department",
        "ent

In [ ]:
{
  "batch_id": "batch_20261016_111604_ccf831f3",
  "canonical_records": {
    "Department": [
      {
        "entity_type": "Department",
        "entity_id": "DEPT-ENG",
        "display_name": "Engineering",
        "properties": {
          "department_id": "DEPT-ENG",
          "department_name": "Engineering",
          "mission": "Build and operate reliable product software",
          "head_employee_id": "EMP-0001",
          "created_at": "2025-01-01T09:00:00Z"
        },
        "foreign_keys": {
          "head_employee_id": [
            "Employee",
            "employee_id",
            "EMP-0001"
          ]
        },
        "source_file": "departments.csv",
        "row_hash": "64773914a0c7c31f",
        "chunk_index": 0,
        "change_type": "added",
        "previous_hash": null,
        "changed_fields": [],
        "previous_values": {},
        "unresolved_fks": {},
        "is_tombstone": false
      },
      {
        "entity_type": "Department",
        "entity_id": "DEPT-PROD",
        "display_name": "Product",
        "properties": {
          "department_id": "DEPT-PROD",
          "department_name": "Product",
          "mission": "Define customer problems, roadmap, and product outcomes",
          "head_employee_id": "EMP-0002",
          "created_at": "2025-01-01T09:00:00Z"
        },
        "foreign_keys": {
          "head_employee_id": [
            "Employee",
            "employee_id",
            "EMP-0002"
          ]
        },
        "source_file": "departments.csv",
        "row_hash": "d11cbc4fe62921b0",
        "chunk_index": 0,
        "change_type": "added",
        "previous_hash": null,
        "changed_fields": [],
        "previous_values": {},
        "unresolved_fks": {},
        "is_tombstone": false
      },
      {
        "entity_type": "Department",
        "entity_id": "DEPT-DATA",
        "display_name": "Data & AI",
        "properties": {
          "department_id": "DEPT-DATA",
          "department_name": "Data & AI",
          "mission": "Build analytics, ML, and AI-powered capabilities",
          "head_employee_id": "EMP-0003",
          "created_at": "2025-01-01T09:00:00Z"
        },
        "foreign_keys": {
          "head_employee_id": [
            "Employee",
            "employee_id",
            "EMP-0003"
          ]
        },
        "source_file": "departments.csv",
        "row_hash": "9445dad2d5af98c5",
        "chunk_index": 0,
        "change_type": "added",
        "previous_hash": null,
        "changed_fields": [],
        "previous_values": {},
        "unresolved_fks": {},
        "is_tombstone": false
      },
      {
        "entity_type": "Department",
        "entity_id": "DEPT-DESIGN",
        "display_name": "Design & Research",
        "properties": {
          "department_id": "DEPT-DESIGN",
          "department_name": "Design & Research",
          "mission": "Create accessible, usable product experiences",
          "head_employee_id": "EMP-0004",
          "created_at": "2025-01-01T09:00:00Z"
        },
        "foreign_keys": {
          "head_employee_id": [
            "Employee",
            "employee_id",
            "EMP-0004"
          ]
        },
        "source_file": "departments.csv",
        "row_hash": "ee43309b9fe87755",
        "chunk_index": 0,
        "change_type": "added",
        "previous_hash": null,
        "changed_fields": [],
        "previous_values": {},
        "unresolved_fks": {},
        "is_tombstone": false
      },
      {
        "entity_type": "Department",
        "entity_id": "DEPT-SALES",
        "display_name": "Sales",
        "properties": {
          "department_id": "DEPT-SALES",
          "department_name": "Sales",
          "mission": "Acquire customers and manage the sales pipeline",
          "head_employee_id": "EMP-0005",
          "created_at": "2025-01-01T09:00:00Z"
        },
        "foreign_keys": {
          "head_employee_id": [
            "Employee",
            "employee_id",
            "EMP-0005"
          ]
        },
        "source_file": "departments.csv",
        "row_hash": "8104a99d1c8c3ad8",
        "chunk_index": 0,
        "change_type": "added",
        "previous_hash": null,
        "changed_fields": [],
        "previous_values": {},
        "unresolved_fks": {},
        "is_tombstone": false
      },
      {
        "entity_type": "Department",
        "entity_id": "DEPT-MKT",
        "display_name": "Marketing",
        "properties": {
          "department_id": "DEPT-MKT",
          "department_name": "Marketing",
          "mission": "Position the product and generate qualified demand",
          "head_employee_id": "EMP-0006",
          "created_at": "2025-01-01T09:00:00Z"
        },
        "foreign_keys": {
          "head_employee_id": [
            "Employee",
            "employee_id",
            "EMP-0006"
          ]
        },
        "source_file": "departments.csv",
        "row_hash": "ca9b7bc1c200419b",
        "chunk_index": 0,
        "change_type": "added",
        "previous_hash": null,
        "changed_fields": [],
        "previous_values": {},
        "unresolved_fks": {},
        "is_tombstone": false
      },
      {
        "entity_type": "Department",
        "entity_id": "DEPT-CS",
        "display_name": "Customer Success",
        "properties": {
          "department_id": "DEPT-CS",
          "department_name": "Customer Success",
          "mission": "Drive adoption, retention, and customer outcomes",
          "head_employee_id": "EMP-0007",
          "created_at": "2025-01-01T09:00:00Z"
        },
        "foreign_keys": {
          "head_employee_id": [
            "Employee",
            "employee_id",
            "EMP-0007"
          ]
        },
        "source_file": "departments.csv",
        "row_hash": "eed4d418b29a185e",
        "chunk_index": 0,
        "change_type": "added",
        "previous_hash": null,
        "changed_fields": [],
        "previous_values": {},
        "unresolved_fks": {},
        "is_tombstone": false
      },
      {
        "entity_type": "Department",
        "entity_id": "DEPT-SUPPORT",
        "display_name": "Customer Support",
        "properties": {
          "department_id": "DEPT-SUPPORT",
          "department_name": "Customer Support",
          "mission": "Resolve customer questions and product issues",
          "head_employee_id": "EMP-0008",
          "created_at": "2025-01-01T09:00:00Z"
        },
        "foreign_keys": {
          "head_employee_id": [
            "Employee",
            "employee_id",
            "EMP-0008"
          ]
        },
        "source_file": "departments.csv",
        "row_hash": "0734d09842bd3104",
        "chunk_index": 0,
        "change_type": "added",
        "previous_hash": null,
        "changed_fields": [],
        "previous_values": {},
        "unresolved_fks": {},
        "is_tombstone": false
      },
      {
        "entity_type": "Department",
        "entity_id": "DEPT-OPS",
        "display_name": "Business Operations",
        "properties": {
          "department_id": "DEPT-OPS",
          "department_name": "Business Operations",
          "mission": "Improve internal processes and operating cadence",
          "head_employee_id": "EMP-0009",
          "created_at": "2025-01-01T09:00:00Z"
        },
        "foreign_keys": {
          "head_employee_id": [
            "Employee",
            "employee_id",
            "EMP-0009"
          ]
        },
        "source_file": "departments.csv",
        "row_hash": "e42f3ff95f3475a4",
        "chunk_index": 0,
        "change_type": "added",
        "previous_hash": null,
        "changed_fields": [],
        "previous_values": {},
        "unresolved_fks": {},
        "is_tombstone": false
      },
      {
        "entity_type": "Department",
        "entity_id": "DEPT-PEOPLE",
        "display_name": "People & Talent",
        "properties": {
          "department_id": "DEPT-PEOPLE",
          "department_name": "People & Talent",
          "mission": "Recruit, develop, and support employees",
          "head_employee_id": "EMP-0010",
          "created_at": "2025-01-01T09:00:00Z"
        },
        "foreign_keys": {
          "head_employee_id": [
            "Employee",
            "employee_id",
            "EMP-0010"
          ]
        },
        "source_file": "departments.csv",
        "row_hash": "52e0bdd494b38efb",
        "chunk_index": 0,
        "change_type": "added",
        "previous_hash": null,
        "changed_fields": [],
        "previous_values": {},
        "unresolved_fks": {},
        "is_tombstone": false
      },
      {
        "entity_type": "Department",
        "entity_id": "DEPT-FIN",
        "display_name": "Finance & Legal",
        "properties": {
          "department_id": "DEPT-FIN",
          "department_name": "Finance & Legal",
          "mission": "Manage finance, contracts, and compliance",
          "head_employee_id": "EMP-0011",
          "created_at": "2025-01-01T09:00:00Z"
        },
        "foreign_keys": {
          "head_employee_id": [
            "Employee",
            "employee_id",
            "EMP-0011"
          ]
        },
        "source_file": "departments.csv",
        "row_hash": "b7f1651f12206a51",
        "chunk_index": 0,
        "change_type": "added",
        "previous_hash": null,
        "changed_fields": [],
        "previous_values": {},
        "unresolved_fks": {},
        "is_tombstone": false
      },
      {
        "entity_type": "Department",
        "entity_id": "DEPT-SEC",
        "display_name": "Security & IT",
        "properties": {
          "department_id": "DEPT-SEC",
          "department_name": "Security & IT",
          "mission": "Protect company systems, access, and data",
          "head_employee_id": "EMP-0012",
          "created_at": "2025-01-01T09:00:00Z"
        },
        "foreign_keys": {
          "head_employee_id": [
            "Employee",
            "employee_id",
            "EMP-0012"
          ]
        },
        "source_file": "departments.csv",
        "row_hash": "01bb4bcf97ef8caa",
        "chunk_index": 0,
        "change_type": "added",
        "previous_hash": null,
        "changed_fields": [],
        "previous_values": {},
        "unresolved_fks": {},
        "is_tombstone": false
      }
    ]
  },
  "relationships_records": [],
  "current_hashes": {
    "DEPT-ENG": "64773914a0c7c31f",
    "DEPT-PROD": "d11cbc4fe62921b0",
    "DEPT-DATA": "9445dad2d5af98c5",
    "DEPT-DESIGN": "ee43309b9fe87755",
    "DEPT-SALES": "8104a99d1c8c3ad8",
    "DEPT-MKT": "ca9b7bc1c200419b",
    "DEPT-CS": "eed4d418b29a185e",
    "DEPT-SUPPORT": "0734d09842bd3104",
    "DEPT-OPS": "e42f3ff95f3475a4",
    "DEPT-PEOPLE": "52e0bdd494b38efb",
    "DEPT-FIN": "b7f1651f12206a51",
    "DEPT-SEC": "01bb4bcf97ef8caa"
  },
  "changed_entities": {
    "added": [
      "DEPT-ENG",
      "DEPT-PROD",
      "DEPT-DATA",
      "DEPT-DESIGN",
      "DEPT-SALES",
      "DEPT-MKT",
      "DEPT-CS",
      "DEPT-SUPPORT",
      "DEPT-OPS",
      "DEPT-PEOPLE",
      "DEPT-FIN",
      "DEPT-SEC"
    ],
    "updated": [],
    "unchanged": []
  },
  "quarantined_records": [],
  "unresolved_foreign_keys": {},
  "now_resolved_foreign_keys": {},
  "chunk_metadata": [
    {
      "filename": "departments.csv",
      "label": "Department",
      "total_rows": 12,
      "chunks": 1,
      "added": 12,
      "updated": 0,
      "unchanged": 0,
      "quarantined": 0
    }
  ],
  "stats": {
    "batch_id": "batch_20261016_111604_ccf831f3",
    "total_files": 1,
    "total_canonical_entities": 12,
    "total_relationships": 0,
    "total_chunks": 1,
    "added": 12,
    "updated": 0,
    "unchanged": 0,
    "quarantined": 0
  },
  "errors": [],
  "stage": "data_collected"
}